# Comparison with Observations: Survey Populations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import pathlib
from scipy.integrate import quad
from scipy import stats

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import utilities.plot_settings

In [ ]:
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * cfg["NS_mass"] * cfg["NS_radius"] ** 2

# Auxiliary quantity beta as defined in eq. (72) of Pons & Vigano (2019).
beta = np.pi**2 * cfg["NS_radius"] ** 6 / (NS_inertia * const.C**3)

# Assume an inclination angle in [rad].
chi = 0.0

# Incorporate the inclination angle dependence into a constant.
beta_1 = beta * (
    cfg["k_coefficients"][0] + cfg["k_coefficients"][1] * np.sin(chi) ** 2
)

## Load observed data

### Load the ATNF Pulsar Catalogue for the observed radio surveys.

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../../data/observations/atnf_full_nobinary_25-03-2025_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.columns

df_atnf.columns = df_atnf.columns.droplevel(1)

In [ ]:
# Select only stars that are not in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]

df_atnf = df_atnf[~df_atnf["ASSOC"].str.match("|".join(discard))]

# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"].to_numpy().astype(np.float64) > 0.01]

df_atnf = df_atnf[
    (df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"].isin(["NAN"]))
]

In [ ]:
# Parks multibeam pulsar survey database.
df_atnf_pk = df_atnf[df_atnf["SURVEY"].str.contains("pksmb")]

# Database of pulsars detected by Parkes multibeam
# that have a pulse width W10 measurement available.
df_atnf_pk_w10 = df_atnf_pk[~df_atnf_pk["W10"].isin(["NAN"])]

# Database of pulsars detected by Parkes multibeam
# that have a proper motion measurement available.
df_atnf_pk_pm = df_atnf_pk[
    (df_atnf_pk["PMRA"] != "NAN") & (df_atnf_pk["PMDEC"] != "NAN")
]

RA_pk_obs = df_atnf_pk["RAJD"].to_numpy().astype(np.float64)
DEC_pk_obs = df_atnf_pk["DECJD"].to_numpy().astype(np.float64)
l_pk_obs = df_atnf_pk["Gl"].to_numpy().astype(np.float64)
b_pk_obs = df_atnf_pk["Gb"].to_numpy().astype(np.float64)
P_pk_obs = df_atnf_pk["P0"].to_numpy().astype(np.float64)
Pdot_pk_obs = df_atnf_pk["P1"].to_numpy().astype(np.float64)
DM_pk_obs = df_atnf_pk["DM"].to_numpy().astype(np.float64)
dist_pk_obs = df_atnf_pk["DIST"].to_numpy().astype(np.float64)
S1400_pk_obs = df_atnf_pk["S1400"].to_numpy().astype(np.float64)

l_pk_obs_pm = df_atnf_pk_pm["Gl"].to_numpy().astype(np.float64)
b_pk_obs_pm = df_atnf_pk_pm["Gb"].to_numpy().astype(np.float64)
Pdot_pk_obs_pm = df_atnf_pk_pm["P1"].to_numpy().astype(np.float64)
dist_pk_obs_pm = df_atnf_pk_pm["DIST"].to_numpy().astype(np.float64)
pmRA_pk_obs = df_atnf_pk_pm["PMRA"].to_numpy().astype(np.float64)
pmDEC_pk_obs = df_atnf_pk_pm["PMDEC"].to_numpy().astype(np.float64)

l_pk_obs_w10 = df_atnf_pk_w10["Gl"].to_numpy().astype(np.float64)
b_pk_obs_w10 = df_atnf_pk_w10["Gb"].to_numpy().astype(np.float64)
P_pk_obs_w10 = df_atnf_pk_w10["P0"].to_numpy().astype(np.float64)
Pdot_pk_obs_w10 = df_atnf_pk_w10["P1"].to_numpy().astype(np.float64)
dist_pk_obs_w10 = df_atnf_pk_w10["DIST"].to_numpy().astype(np.float64)
w10_pk_obs = df_atnf_pk_w10["W10"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_pk_obs[(l_pk_obs > 180.0) & (l_pk_obs < 360.0)] = (
    l_pk_obs[(l_pk_obs > 180.0) & (l_pk_obs < 360.0)] - 360.0
)
l_pk_obs_pm[(l_pk_obs_pm > 180.0) & (l_pk_obs_pm < 360.0)] = (
    l_pk_obs_pm[(l_pk_obs_pm > 180.0) & (l_pk_obs_pm < 360.0)] - 360.0
)
l_pk_obs_w10[(l_pk_obs_w10 > 180.0) & (l_pk_obs_w10 < 360.0)] = (
    l_pk_obs_w10[(l_pk_obs_w10 > 180.0) & (l_pk_obs_w10 < 360.0)] - 360.0
)

# Select only pulsars falling in the Parkes multibeam sky coverage where completness is above 90%.
cond = (l_pk_obs > -100.0) & (l_pk_obs < 50.0) & (np.abs(b_pk_obs) < 5.0)
cond_w10 = (
    (l_pk_obs_w10 > -100.0)
    & (l_pk_obs_w10 < 50.0)
    & (np.abs(b_pk_obs_w10) < 5.0)
)
cond_pm = (
    (l_pk_obs_pm > -100.0) & (l_pk_obs_pm < 50.0) & (np.abs(b_pk_obs_pm) < 5.0)
)

RA_pk_obs = RA_pk_obs[cond]
DEC_pk_obs = DEC_pk_obs[cond]
l_pk_obs = l_pk_obs[cond]
b_pk_obs = b_pk_obs[cond]
P_pk_obs = P_pk_obs[cond]
Pdot_pk_obs = Pdot_pk_obs[cond]
DM_pk_obs = DM_pk_obs[cond]
dist_pk_obs = dist_pk_obs[cond]
S1400_pk_obs = S1400_pk_obs[cond]
P_pk_obs_w10 = P_pk_obs_w10[cond_w10]
w10_pk_obs = w10_pk_obs[cond_w10]
pmRA_pk_obs = pmRA_pk_obs[cond_pm]
pmDEC_pk_obs = pmDEC_pk_obs[cond_pm]

number_pk = len(RA_pk_obs)

In [ ]:
# Swinburne multibeam pulsar survey database.
df_atnf_sw = df_atnf[df_atnf["SURVEY"].str.contains("pkssw")]

# Database of pulsars detected by Swinburne that have a pulse width W10 measurment available.
df_atnf_sw_w10 = df_atnf_sw[~df_atnf_sw["W10"].isin(["NAN"])]
# Database of pulsars detected by Swinburne that have a proper motion measurment available.
df_atnf_sw_pm = df_atnf_sw[
    (df_atnf_sw["PMRA"] != "NAN") & (df_atnf_sw["PMDEC"] != "NAN")
]

RA_sw_obs = df_atnf_sw["RAJD"].to_numpy().astype(np.float64)
DEC_sw_obs = df_atnf_sw["DECJD"].to_numpy().astype(np.float64)
l_sw_obs = df_atnf_sw["Gl"].to_numpy().astype(np.float64)
b_sw_obs = df_atnf_sw["Gb"].to_numpy().astype(np.float64)
P_sw_obs = df_atnf_sw["P0"].to_numpy().astype(np.float64)
Pdot_sw_obs = df_atnf_sw["P1"].to_numpy().astype(np.float64)
DM_sw_obs = df_atnf_sw["DM"].to_numpy().astype(np.float64)
dist_sw_obs = df_atnf_sw["DIST"].to_numpy().astype(np.float64)
S1400_sw_obs = df_atnf_sw["S1400"].to_numpy().astype(np.float64)

l_sw_obs_pm = df_atnf_sw_pm["Gl"].to_numpy().astype(np.float64)
b_sw_obs_pm = df_atnf_sw_pm["Gb"].to_numpy().astype(np.float64)
Pdot_sw_obs_pm = df_atnf_sw_pm["P1"].to_numpy().astype(np.float64)
dist_sw_obs_pm = df_atnf_sw_pm["DIST"].to_numpy().astype(np.float64)
pmRA_sw_obs = df_atnf_sw_pm["PMRA"].to_numpy().astype(np.float64)
pmDEC_sw_obs = df_atnf_sw_pm["PMDEC"].to_numpy().astype(np.float64)

l_sw_obs_w10 = df_atnf_sw_w10["Gl"].to_numpy().astype(np.float64)
b_sw_obs_w10 = df_atnf_sw_w10["Gb"].to_numpy().astype(np.float64)
P_sw_obs_w10 = df_atnf_sw_w10["P0"].to_numpy().astype(np.float64)
Pdot_sw_obs_w10 = df_atnf_sw_w10["P1"].to_numpy().astype(np.float64)
dist_sw_obs_w10 = df_atnf_sw_w10["DIST"].to_numpy().astype(np.float64)
w10_sw_obs = df_atnf_sw_w10["W10"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_sw_obs[(l_sw_obs > 180.0) & (l_sw_obs < 360.0)] = (
    l_sw_obs[(l_sw_obs > 180.0) & (l_sw_obs < 360.0)] - 360.0
)
l_sw_obs_pm[(l_sw_obs_pm > 180.0) & (l_sw_obs_pm < 360.0)] = (
    l_sw_obs_pm[(l_sw_obs_pm > 180.0) & (l_sw_obs_pm < 360.0)] - 360.0
)
l_sw_obs_w10[(l_sw_obs_w10 > 180.0) & (l_sw_obs_w10 < 360.0)] = (
    l_sw_obs_w10[(l_sw_obs_w10 > 180.0) & (l_sw_obs_w10 < 360.0)] - 360.0
)

# Selection only pulsars falling in the Swinburne sky coverage where completness is above 90%.
cond = (l_sw_obs > -100.0) & (l_sw_obs < 50.0)
cond_w10 = (l_sw_obs_w10 > -100.0) & (l_sw_obs_w10 < 50.0)
cond_pm = (l_sw_obs_pm > -100.0) & (l_sw_obs_pm < 50.0)

RA_sw_obs = RA_sw_obs[cond]
DEC_sw_obs = DEC_sw_obs[cond]
l_sw_obs = l_sw_obs[cond]
b_sw_obs = b_sw_obs[cond]
P_sw_obs = P_sw_obs[cond]
Pdot_sw_obs = Pdot_sw_obs[cond]
DM_sw_obs = DM_sw_obs[cond]
dist_sw_obs = dist_sw_obs[cond]
S1400_sw_obs = S1400_sw_obs[cond]
P_sw_obs_w10 = P_sw_obs_w10[cond_w10]
w10_sw_obs = w10_sw_obs[cond_w10]
pmRA_sw_obs = pmRA_sw_obs[cond_pm]
pmDEC_sw_obs = pmDEC_sw_obs[cond_pm]

number_sw = len(RA_sw_obs)

In [ ]:
# HTRU pulsar survey database.
df_atnf_htru = df_atnf[df_atnf["SURVEY"].str.contains("htru_pks")]

# Database of pulsars detected by Swinburne that have a pulse width W10 measurment available.
df_atnf_htru_w10 = df_atnf_htru[~df_atnf_htru["W10"].isin(["NAN"])]
# Database of pulsars detected by Swinburne that have a proper motion measurment available.
df_atnf_htru_pm = df_atnf_htru[
    (df_atnf_htru["PMRA"] != "NAN") & (df_atnf_htru["PMDEC"] != "NAN")
]

RA_htru_obs = df_atnf_htru["RAJD"].to_numpy().astype(np.float64)
DEC_htru_obs = df_atnf_htru["DECJD"].to_numpy().astype(np.float64)
l_htru_obs = df_atnf_htru["Gl"].to_numpy().astype(np.float64)
b_htru_obs = df_atnf_htru["Gb"].to_numpy().astype(np.float64)
P_htru_obs = df_atnf_htru["P0"].to_numpy().astype(np.float64)
Pdot_htru_obs = df_atnf_htru["P1"].to_numpy().astype(np.float64)
DM_htru_obs = df_atnf_htru["DM"].to_numpy().astype(np.float64)
dist_htru_obs = df_atnf_htru["DIST"].to_numpy().astype(np.float64)
S1400_htru_obs = df_atnf_htru["S1400"].to_numpy().astype(np.float64)

l_htru_obs_pm = df_atnf_htru_pm["Gl"].to_numpy().astype(np.float64)
b_htru_obs_pm = df_atnf_htru_pm["Gb"].to_numpy().astype(np.float64)
Pdot_htru_obs_pm = df_atnf_htru_pm["P1"].to_numpy().astype(np.float64)
dist_htru_obs_pm = df_atnf_htru_pm["DIST"].to_numpy().astype(np.float64)
pmRA_htru_obs = df_atnf_htru_pm["PMRA"].to_numpy().astype(np.float64)
pmDEC_htru_obs = df_atnf_htru_pm["PMDEC"].to_numpy().astype(np.float64)

l_htru_obs_w10 = df_atnf_htru_w10["Gl"].to_numpy().astype(np.float64)
b_htru_obs_w10 = df_atnf_htru_w10["Gb"].to_numpy().astype(np.float64)
P_htru_obs_w10 = df_atnf_htru_w10["P0"].to_numpy().astype(np.float64)
Pdot_htru_obs_w10 = df_atnf_htru_w10["P1"].to_numpy().astype(np.float64)
dist_htru_obs_w10 = df_atnf_htru_w10["DIST"].to_numpy().astype(np.float64)
w10_htru_obs = df_atnf_htru_w10["W10"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] = (
    l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] - 360.0
)
l_htru_obs_pm[(l_htru_obs_pm > 180.0) & (l_htru_obs_pm < 360.0)] = (
    l_htru_obs_pm[(l_htru_obs_pm > 180.0) & (l_htru_obs_pm < 360.0)] - 360.0
)
l_htru_obs_w10[(l_htru_obs_w10 > 180.0) & (l_htru_obs_w10 < 360.0)] = (
    l_htru_obs_w10[(l_htru_obs_w10 > 180.0) & (l_htru_obs_w10 < 360.0)] - 360.0
)

# Selection only pulsars falling in the HTRU sky coverage where completness is above 90%.
cond = (
    (l_htru_obs > -120.0) & (l_htru_obs < 30.0) & (np.abs(b_htru_obs) < 15.0)
)
cond_pm = (
    (l_htru_obs_pm > -120.0)
    & (l_htru_obs_pm < 30.0)
    & (np.abs(b_htru_obs_pm) < 15.0)
)
cond_w10 = (
    (l_htru_obs_w10 > -120.0)
    & (l_htru_obs_w10 < 30.0)
    & (np.abs(b_htru_obs_w10) < 15.0)
)

RA_htru_obs = RA_htru_obs[cond]
DEC_htru_obs = DEC_htru_obs[cond]
l_htru_obs = l_htru_obs[cond]
b_htru_obs = b_htru_obs[cond]
P_htru_obs = P_htru_obs[cond]
Pdot_htru_obs = Pdot_htru_obs[cond]
DM_htru_obs = DM_htru_obs[cond]
dist_htru_obs = dist_htru_obs[cond]
S1400_htru_obs = S1400_htru_obs[cond]
P_htru_obs_w10 = P_htru_obs_w10[cond_w10]
w10_htru_obs = w10_htru_obs[cond_w10]
pmRA_htru_obs = pmRA_htru_obs[cond_pm]
pmDEC_htru_obs = pmDEC_htru_obs[cond_pm]

number_htru = len(RA_htru_obs)

In [ ]:
RA_all_obs = np.concatenate((RA_pk_obs, RA_sw_obs, RA_htru_obs))
DEC_all_obs = np.concatenate((DEC_pk_obs, DEC_sw_obs, DEC_htru_obs))
pmRA_all_obs = np.concatenate((pmRA_pk_obs, pmRA_sw_obs, pmRA_htru_obs))
pmDEC_all_obs = np.concatenate((pmDEC_pk_obs, pmDEC_sw_obs, pmDEC_htru_obs))
l_all_obs = np.concatenate((l_pk_obs, l_sw_obs, l_htru_obs))
b_all_obs = np.concatenate((b_pk_obs, b_sw_obs, b_htru_obs))
P_all_obs = np.concatenate((P_pk_obs, P_sw_obs, P_htru_obs))
Pdot_all_obs = np.concatenate((Pdot_pk_obs, Pdot_sw_obs, Pdot_htru_obs))
DM_all_obs = np.concatenate((DM_pk_obs, DM_sw_obs, DM_htru_obs))
dist_all_obs = np.concatenate((dist_pk_obs, dist_sw_obs, dist_htru_obs))
S1400_all_obs = np.concatenate((S1400_pk_obs, S1400_sw_obs, S1400_htru_obs))
P_all_obs_w10 = np.concatenate((P_pk_obs_w10, P_sw_obs_w10, P_htru_obs_w10))
w10_all_obs = np.concatenate((w10_pk_obs, w10_sw_obs, w10_htru_obs))

In [ ]:
print(f"Number of pulsars detected by Parks multibeam: {number_pk}")
print(f"Number of pulsars detected by Swinburne: {number_sw}")
print(f"Number of pulsars detected by HTRU: {number_htru}")

## Load simulated data

### Load the output of a simulated detection with the radio and X-ray surveys.

In [ ]:
path_to_simulation = pathlib.Path("../../data/example_simulation_magrot_det/")

# Load the `.pkl.gz` files containing the survey results to import.
df_PMPS_sim = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
df_PMPS_sim.head()

df_SMPS_sim = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)

df_HTRU_sim = pd.read_pickle(
    pathlib.Path().joinpath(
        path_to_simulation, "survey_HTRU_low_mid_results.pkl.gz"
    ),
    compression="gzip",
)

In [ ]:
# Extracting the parameters.
RA_pk_sim = df_PMPS_sim["ra"]["[deg]"].to_numpy()
DEC_pk_sim = df_PMPS_sim["dec"]["[deg]"].to_numpy()
l_pk_sim = df_PMPS_sim["l"]["[deg]"].to_numpy()
b_pk_sim = df_PMPS_sim["b"]["[deg]"].to_numpy()
pmRA_pk_sim = df_PMPS_sim["pm_ra"]["[mas yr^-1]"].to_numpy()
pmDEC_pk_sim = df_PMPS_sim["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_pk_sim = df_PMPS_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_pk_sim = df_PMPS_sim["dist"]["[kpc]"].to_numpy()
P_pk_sim = df_PMPS_sim["P"]["[s]"].to_numpy()
Pdot_pk_sim = df_PMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_pk_sim = df_PMPS_sim["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_pk_sim = df_PMPS_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
RA_sw_sim = df_SMPS_sim["ra"]["[deg]"].to_numpy()
DEC_sw_sim = df_SMPS_sim["dec"]["[deg]"].to_numpy()
l_sw_sim = df_SMPS_sim["l"]["[deg]"].to_numpy()
b_sw_sim = df_SMPS_sim["b"]["[deg]"].to_numpy()
pmRA_sw_sim = df_SMPS_sim["pm_ra"]["[mas yr^-1]"].to_numpy()
pmDEC_sw_sim = df_SMPS_sim["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_sw_sim = df_SMPS_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_sw_sim = df_SMPS_sim["dist"]["[kpc]"].to_numpy()
P_sw_sim = df_SMPS_sim["P"]["[s]"].to_numpy()
Pdot_sw_sim = df_SMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_sw_sim = df_SMPS_sim["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_sw_sim = df_SMPS_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
RA_htru_sim = df_HTRU_sim["ra"]["[deg]"].to_numpy()
DEC_htru_sim = df_HTRU_sim["dec"]["[deg]"].to_numpy()
l_htru_sim = df_HTRU_sim["l"]["[deg]"].to_numpy()
b_htru_sim = df_HTRU_sim["b"]["[deg]"].to_numpy()
pmRA_htru_sim = df_HTRU_sim["pm_ra"]["[mas yr^-1]"].to_numpy()
pmDEC_htru_sim = df_HTRU_sim["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_htru_sim = df_HTRU_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_htru_sim = df_HTRU_sim["dist"]["[kpc]"].to_numpy()
P_htru_sim = df_HTRU_sim["P"]["[s]"].to_numpy()
Pdot_htru_sim = df_HTRU_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_htru_sim = df_HTRU_sim["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_htru_sim = df_HTRU_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
print(
    f"Number of pulsars detected by the simulated Parkes multibeam survey: {len(RA_pk_sim)}"
)
print(
    f"Number of pulsars detected by the simulated Swinburne survey: {len(RA_sw_sim)}"
)
print(
    f"Number of pulsars detected by the simulated HTRU surveys: {len(RA_htru_sim)}"
)
print(
    f"Number of pulsars detected by the simulated HTRU low survey: {len(df_HTRU_sim.loc[(df_HTRU_sim['HTRU_low'] ==1) ])}"
)
print(
    f"Number of pulsars detected by the simulated HTRU mid survey: {len(df_HTRU_sim.loc[(df_HTRU_sim['HTRU_mid'] ==1) ])}"
)

## Compare simulations with observations

Comparison of the sky distributions.

In [ ]:
RA_galcen = 266.4
DEC_galcen = -29.0

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    RA_pk_obs,
    DEC_pk_obs,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    RA_sw_obs,
    DEC_sw_obs,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)

ax.plot(
    RA_htru_obs,
    DEC_htru_obs,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed HTRU",
)

ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    RA_pk_sim,
    DEC_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    RA_sw_sim,
    DEC_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated SMPS",
)

ax.plot(
    RA_htru_sim,
    DEC_htru_sim,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_pk_obs,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    RA_pk_sim,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)


ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0.0, 360.0)
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_sw_obs,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    RA_sw_sim,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0.0, 360.0)
ax.legend(frameon=True, loc=2)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_htru_obs,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    RA_htru_sim,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0.0, 360.0)
ax.legend(frameon=True, loc=2)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_pk_obs,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    DEC_pk_sim,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_sw_obs,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    DEC_sw_sim,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_htru_obs,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    DEC_htru_sim,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l_pk_obs,
    b_pk_obs,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    l_sw_obs,
    b_sw_obs,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    l_htru_obs,
    b_htru_obs,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed HTRU",
)

ax.plot(0.0, 0.0, marker="*", color="tab:blue", markersize=20)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l_pk_sim,
    b_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    l_sw_sim,
    b_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    l_htru_sim,
    b_htru_sim,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.plot(0.0, 0.0, marker="*", color="tab:blue", markersize=20)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
l_edges = np.linspace(-180.0, 180.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    l_pk_obs,
    bins=l_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    l_pk_sim,
    bins=l_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"l [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-180.0, 180.0)
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    l_sw_obs,
    bins=l_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    l_sw_sim,
    bins=l_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"l [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-180.0, 180.0)
ax.legend(frameon=True, loc=2)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    l_htru_obs,
    bins=l_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    l_htru_sim,
    bins=l_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"l [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-180.0, 180.0)
ax.legend(frameon=True, loc=2)

plt.show()

In [ ]:
b_edges = np.linspace(-40.0, 40.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    b_pk_obs,
    bins=b_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    b_pk_sim,
    bins=b_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"b [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-40.0, 40.0)
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    b_sw_obs,
    bins=b_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    b_sw_sim,
    bins=b_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"b [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-40.0, 40.0)
ax.legend(frameon=True, loc=2)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    b_htru_obs,
    bins=b_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    b_htru_sim,
    bins=b_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"b [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-40.0, 40.0)
ax.legend(frameon=True, loc=2)

plt.show()

Compare proper motion distributions.

In [ ]:
pmRA_edges = np.linspace(-90.0, 90.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmRA_pk_obs,
    bins=pmRA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    pmRA_pk_sim,
    bins=pmRA_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmRA_sw_obs,
    bins=pmRA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    pmRA_sw_sim,
    bins=pmRA_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmRA_htru_obs,
    bins=pmRA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    pmRA_htru_sim,
    bins=pmRA_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
pmDEC_edges = np.linspace(-90.0, 90.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_pk_obs,
    bins=pmDEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    pmDEC_pk_sim,
    bins=pmDEC_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_sw_obs,
    bins=pmDEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    pmDEC_sw_sim,
    bins=pmDEC_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_htru_obs,
    bins=pmDEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    pmDEC_htru_sim,
    bins=pmDEC_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

Compare DM distributions.

In [ ]:
dm_edges = np.linspace(0, 2500, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    DM_pk_obs,
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed PMPS",
    rasterized=True,
)
ax.hist(
    DM_pk_sim,
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Simulated PMPS",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
# ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    DM_sw_obs,
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed SMPS",
    rasterized=True,
)
ax.hist(
    DM_sw_sim,
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.0,
    label=r"Simulated SMPS",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
# ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    DM_htru_obs,
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed HTRU",
    rasterized=True,
)
ax.hist(
    DM_htru_sim,
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1.0,
    label=r"Simulated HTRU",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
# ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

Compare distance distributions.

In [ ]:
d_edges = np.linspace(0, 30, 36)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_pk_obs,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed PMPS",
    rasterized=True,
)
ax.hist(
    dist_pk_sim,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Simulated PMPS",
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_sw_obs,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed SMPS",
    rasterized=True,
)
ax.hist(
    dist_sw_sim,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.0,
    label=r"Simulated SMPS",
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.legend(frameon=True, loc=0)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_htru_obs,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed HTRU",
    rasterized=True,
)
ax.hist(
    dist_htru_sim,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1.0,
    label=r"Simulated HTRU",
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.legend(frameon=True, loc=0)

plt.show()

Comparing the spin-period and spin-period-derivative distributions.

In [ ]:
P_bins = np.logspace(-2.0, 2.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_pk_obs,
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    P_pk_sim,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_sw_obs,
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    P_sw_sim,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_htru_obs,
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    P_htru_sim,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
Pdot_bins = np.logspace(-20.0, -8.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Pdot_pk_obs,
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    Pdot_pk_sim,
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)

plt.xlabel(r"$\dot{P}$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Pdot_sw_obs,
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    Pdot_sw_sim,
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)

plt.xlabel(r"$\dot{P}$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Pdot_htru_obs,
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    Pdot_htru_sim,
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)

plt.xlabel(r"$\dot{P}$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

Comparing PPdot diagrams.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_obs,
    Pdot_pk_obs,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    P_sw_obs,
    Pdot_sw_obs,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    P_htru_obs,
    Pdot_htru_obs,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Observed HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-3, 100.0)
ax.set_ylim(1.0e-21, 1.0e-9)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=True, loc=2)

plt.grid()

fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_sim,
    Pdot_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    P_sw_sim,
    Pdot_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    P_htru_sim,
    Pdot_htru_sim,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-3, 100.0)
ax.set_ylim(1.0e-21, 1.0e-9)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=True, loc=0)

plt.grid()

Comparing the spin-down power distributions.

In [ ]:
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * cfg["NS_mass"] * cfg["NS_radius"] ** 2

# Computing the spin-down power.
Erot_dot_pk_sim = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_pk_sim / (P_pk_sim**3)
)
Erot_dot_sw_sim = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_sw_sim / (P_sw_sim**3)
)
Erot_dot_htru_sim = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_htru_sim / (P_htru_sim**3)
)

Erot_dot_pk_obs = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_pk_obs / (P_pk_obs**3)
)
Erot_dot_sw_obs = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_sw_obs / (P_sw_obs**3)
)
Erot_dot_htru_obs = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_htru_obs / (P_htru_obs**3)
)

In [ ]:
Erot_dot_bins = np.logspace(27.0, 40.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot_pk_obs,
    bins=Erot_dot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    Erot_dot_pk_sim,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot_sw_obs,
    bins=Erot_dot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    Erot_dot_sw_sim,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot_htru_obs,
    bins=Erot_dot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    Erot_dot_htru_sim,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

Comparing the characteristic age distributions.

In [ ]:
age_char_pk_obs = P_pk_obs / (2 * Pdot_pk_obs) / const.YR_TO_S
age_char_sw_obs = P_sw_obs / (2 * Pdot_sw_obs) / const.YR_TO_S
age_char_htru_obs = P_htru_obs / (2 * Pdot_htru_obs) / const.YR_TO_S

age_char_pk_sim = P_pk_sim / (2 * Pdot_pk_sim) / const.YR_TO_S
age_char_sw_sim = P_sw_sim / (2 * Pdot_sw_sim) / const.YR_TO_S
age_char_htru_sim = P_htru_sim / (2 * Pdot_htru_sim) / const.YR_TO_S

In [ ]:
age_char_bins = np.logspace(2, 11.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    age_char_pk_obs,
    bins=age_char_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    age_char_pk_sim,
    bins=age_char_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$\tau_{\rm c}$ [yr]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    age_char_sw_obs,
    bins=age_char_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    age_char_sw_sim,
    bins=age_char_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$\tau_{\rm c}$ [yr]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    age_char_htru_obs,
    bins=age_char_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    age_char_htru_sim,
    bins=age_char_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$\tau_{\rm c}$ [yr]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

Comparing the magnetic field distributions assuming inclination angle $\chi = 0$ deg.

In [ ]:
B_pk_obs = np.sqrt(1.0 / beta_1 * P_pk_obs * Pdot_pk_obs)
B_sw_obs = np.sqrt(1.0 / beta_1 * P_sw_obs * Pdot_sw_obs)
B_htru_obs = np.sqrt(1.0 / beta_1 * P_htru_obs * Pdot_htru_obs)

B_pk_sim = np.sqrt(1.0 / beta_1 * P_pk_sim * Pdot_pk_sim)
B_sw_sim = np.sqrt(1.0 / beta_1 * P_sw_sim * Pdot_sw_sim)
B_htru_sim = np.sqrt(1.0 / beta_1 * P_htru_sim * Pdot_htru_sim)

In [ ]:
B_bins = np.logspace(9.0, 17.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    B_pk_obs,
    bins=B_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    B_pk_sim,
    bins=B_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$B$ [G]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    B_sw_obs,
    bins=B_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    B_sw_sim,
    bins=B_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$B$ [G]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    B_htru_obs,
    bins=B_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    B_htru_sim,
    bins=B_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$B$ [G]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

Comparing the radio-flux distributions.

In [ ]:
S_radio_bins = np.logspace(-5, 1, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_pk_obs / 1000.0,
    bins=S_radio_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    S1400_pk_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_sw_obs / 1000.0,
    bins=S_radio_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    S1400_sw_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_htru_obs / 1000.0,
    bins=S_radio_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    S1400_htru_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
L_pseudo_radio_pk_obs = (
    S1400_pk_obs
    * const.MILLIJY_TO_ERG
    * (dist_pk_obs * const.KPC_TO_CM) ** 2
    * 1.374e9
)
L_pseudo_radio_sw_obs = (
    S1400_sw_obs
    * const.MILLIJY_TO_ERG
    * (dist_sw_obs * const.KPC_TO_CM) ** 2
    * 1.374e9
)
L_pseudo_radio_htru_obs = (
    S1400_htru_obs
    * const.MILLIJY_TO_ERG
    * (dist_htru_obs * const.KPC_TO_CM) ** 2
    * 1.374e9
)

L_pseudo_radio_pk_sim = (
    S1400_pk_sim
    * 1000
    * const.MILLIJY_TO_ERG
    * (dist_pk_sim * const.KPC_TO_CM) ** 2
    * 1.374e9
)
L_pseudo_radio_sw_sim = (
    S1400_sw_sim
    * 1000
    * const.MILLIJY_TO_ERG
    * (dist_sw_sim * const.KPC_TO_CM) ** 2
    * 1.374e9
)
L_pseudo_radio_htru_sim = (
    S1400_htru_sim
    * 1000
    * const.MILLIJY_TO_ERG
    * (dist_htru_sim * const.KPC_TO_CM) ** 2
    * 1.374e9
)

eff_radio_pk_obs = L_pseudo_radio_pk_obs / Erot_dot_pk_obs
eff_radio_sw_obs = L_pseudo_radio_sw_obs / Erot_dot_sw_obs
eff_radio_htru_obs = L_pseudo_radio_htru_obs / Erot_dot_htru_obs

eff_radio_pk_sim = L_pseudo_radio_pk_sim / Erot_dot_pk_sim
eff_radio_sw_sim = L_pseudo_radio_sw_sim / Erot_dot_sw_sim
eff_radio_htru_sim = L_pseudo_radio_htru_sim / Erot_dot_htru_sim

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax.set_ylabel(r"Efficiency")

ax.loglog(
    Erot_dot_pk_obs,
    eff_radio_pk_obs,
    "o",
    color="darkgray",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Observed PMPS",
)
ax.loglog(
    Erot_dot_pk_sim,
    eff_radio_pk_sim,
    "o",
    color="tab:red",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Simulated PMPS",
)

ax.axhline(y=1, color="black", linestyle="-", linewidth=2)
ax.axhline(y=0.1, color="black", linestyle="--", linewidth=2)

plt.legend(frameon=True, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax.set_ylabel(r"Efficiency")

ax.loglog(
    Erot_dot_sw_obs,
    eff_radio_sw_obs,
    "o",
    color="darkgray",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Observed SMPS",
)
ax.loglog(
    Erot_dot_sw_sim,
    eff_radio_sw_sim,
    "o",
    color="tab:blue",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Simulated SMPS",
)
ax.axhline(y=1, color="black", linestyle="-", linewidth=2)
ax.axhline(y=0.1, color="black", linestyle="--", linewidth=2)

plt.legend(frameon=True, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax.set_ylabel(r"Efficiency")

ax.loglog(
    Erot_dot_htru_obs,
    eff_radio_htru_obs,
    "o",
    color="darkgray",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Observed HTRU",
)
ax.loglog(
    Erot_dot_htru_sim,
    eff_radio_htru_sim,
    "o",
    color="tab:green",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Simulated HTRU",
)

ax.axhline(y=1, color="black", linestyle="-", linewidth=2)
ax.axhline(y=0.1, color="black", linestyle="--", linewidth=2)

plt.legend(frameon=True, loc=0, fontsize=20)

plt.show()

Comparing the pulse-width distributions.

In [ ]:
# Converting pulse width into [deg].
w_pk_sim_deg = w_eff_pk_sim / P_pk_sim * 360.0
w_sw_sim_deg = w_eff_sw_sim / P_sw_sim * 360.0
w_htru_sim_deg = w_eff_htru_sim / P_htru_sim * 360.0

w10_pk_obs_deg = w10_pk_obs / 1000 / P_pk_obs_w10 * 360.0
w10_sw_obs_deg = w10_sw_obs / 1000 / P_sw_obs_w10 * 360.0
w10_htru_obs_deg = w10_htru_obs / 1000 / P_htru_obs_w10 * 360.0

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.loglog(
    P_pk_obs_w10,
    w10_pk_obs_deg,
    "o",
    color="tab:red",
    ms=6,
    rasterized=True,
    label="Observed PMPS",
)
ax.loglog(
    P_sw_obs_w10,
    w10_sw_obs_deg,
    "o",
    color="tab:blue",
    ms=6,
    rasterized=True,
    label="Observed SMPS",
)

ax.loglog(
    P_htru_obs_w10,
    w10_htru_obs_deg,
    "o",
    fillstyle="none",
    color="tab:green",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Observed HTRU",
)
plt.xlim(3.0e-2, 20)
plt.ylim(1, 400)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [deg]")
ax.legend(frameon=True, loc=3)


fig, ax = plt.subplots(figsize=(15, 8))

ax.loglog(
    P_pk_sim,
    w_pk_sim_deg,
    "o",
    color="tab:red",
    ms=6,
    rasterized=True,
    label="Simulated PMPS",
)
ax.loglog(
    P_sw_sim,
    w_sw_sim_deg,
    "o",
    color="tab:blue",
    ms=6,
    rasterized=True,
    label="Simulated SMPS",
)
ax.loglog(
    P_htru_sim,
    w_htru_sim_deg,
    "o",
    fillstyle="none",
    color="tab:green",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Simulated HTRU",
)

plt.xlim(3.0e-2, 20)
plt.ylim(1, 400)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [deg]")
ax.legend(frameon=True, loc=3)

plt.show()